In [1]:
#######################
# BANK CHURN PREDICTION
#######################
# import libraries
# pandas for data handling
# numpy for numerical operations
# matplotlib and seaborn for visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from IPython.display import display

# ML utilities
# split data, hyperparameter tuning
# feature scaling
# chain pre-processing
# performance evaluation
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

# machine learning models for comparison
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB



In [2]:
########
# DATA
########

# Define categorical columns
CATEGORICAL_COLS = [
    'Gender', 'Education_Level', 'Marital_Status',
    'Income_Category', 'Card_Category'
]

# Data load
def load_data(path):
    
    # If the file cannot be found in the location I specified, display the message below, easy fix
    if not os.path.exists(path):
        raise FileNotFoundError(f"Data file not found: {path}")

    # Read the file using the path provided
    df = pd.read_csv(path)
    
    # Drop first (Client Number) and last two columns (existing NB predictions so they will not be used as predictors)
    df = df.iloc[:, 1:-2]
    
    # The dependent variable is Attrition_Flag
    df['Attrition_Flag'] = df['Attrition_Flag'].map({
        'Existing Customer': 0,
        'Attrited Customer': 1
    })
    
    # One-hot encode categorical columns
    df = pd.get_dummies(df, columns=CATEGORICAL_COLS, drop_first=True)
    
    return df

# Some analytics - creating tables to see the distribution of attrition within variables
def run_eda_tables(df):
    
    tables = {}

    # Specify overall attrition
    # df['Attrition_Flag'] selects my target column, where 0 = Existing Customer and 1 = Attrited Customer
    # value_counts(normalize=True) counts how many of each value there are and normalizes it, i.e., returns proportions
    # Multiplying by 100 converts the proportions into percentages.
    overall = df['Attrition_Flag'].value_counts(normalize=True) * 100
    # pd.DataFrame(overall) converts the series into a data frame
    # .T transposes it so that the segments or buckets become rows, and the two categories (0 and 1) become columns
    # .rename(columns={0: 'Existing (%)', 1: 'Attrited (%)'}) gives meaningful column names instead of just 0 and 1
    overall_table = pd.DataFrame(overall).T.rename(columns={0: 'Existing (%)', 1: 'Attrited (%)'})
    # store the table in the tables dictionary with key 'Overall'.
    # .round(2) rounds percentages to 4 decimal places (most people like 2, I like 4...)
    tables['Overall'] = overall_table.round(4)

    # categorical variables
    # loop through each column in CATEGORICAL_COLS, calculate attrition percentages for each segment or bucket
    # the if statement checks to see if the column exists in the data frame I created above
    # pd.crosstab(df[col], df['Attrition_Flag']) creates a table showing counts of each attrition flag for 
        # every category in col
    # normalize='index' converts counts into percentages row-wise and the values sum to 100%.
    for col in CATEGORICAL_COLS:
        if col in df.columns:
            table = pd.crosstab(df[col], df['Attrition_Flag'], normalize='index') * 100
            tables[col] = table.rename(columns={0: 'Existing (%)', 1: 'Attrited (%)'}).round(2)

    # Numeric variables
    # df.select_dtypes(include=[np.number]) selects all columns with numeric data types 
    # .columns returns just the column names of numeric columns
    # the second line remove Attrition_Flag from the list of numeric columns (this is the target)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    numeric_cols = [col for col in numeric_cols if col != 'Attrition_Flag']

    # Loop through each numeric columns prepared above
    # the goal is to create binned categories for each colums
    
    for col in numeric_cols:
        try:
            # pd.qcut(df[col], q=10) divides the numeric column into 10 quantiles 
            # duplicates='drop' handles cases where some quantile edges are identical (e.g., column has few unique values
            df[f'{col}_bin'] = pd.qcut(df[col], q=10, duplicates='drop')
            # If qcut fails because there are too few unique values to split into 10 quantiles, it falls back to pd.cut
            # pd.cut(df[col], bins=5) divides the numeric column into 5 equal-width bins instead
        except ValueError:  # fallback for few unique values
            df[f'{col}_bin'] = pd.cut(df[col], bins=5)
        
        # Create a cross-tabulation table of segment or bins vs. attrition.
        # normalize='index' converts counts to percentages so each bin shows the proportion of existing vs. attrited 
        # Multiplying by 100 converts proportions to percentages.
        table = pd.crosstab(df[f'{col}_bin'], df['Attrition_Flag'], normalize='index') * 100
        # Rename columns 0 to 'Existing (%)' and 1 to 'Attrited (%)'
        # Rounds percentages to 4 decimal places.
        # Store the result in the tables dictionary under the key of the numeric column name
        tables[col] = table.rename(columns={0: 'Existing (%)', 1: 'Attrited (%)'}).round(4)
        # Remove the temporary bin column from the data frame to keep it clean
        df.drop(columns=f'{col}_bin', inplace=True)

    return tables


#############
# MAIN - Data
#############
def main(path):
    
    # load the input file, drop unnecessary columns, convert Attrition_Flag to 0,1, convert categorical variables
    df = load_data(path)
    # run the function created earlier
    attrition_tables = run_eda_tables(df)

    # Display tables in Jupyter
    for var, table in attrition_tables.items():
        print(f"\n--- {var} ---")
        display(table)
    
    # return tao things - df (data frame for modeling), dictionary of tables generated by run_eda_tables
    return df, attrition_tables


# Run the main workflow automatically from my personal projects folder
df, attrition_tables = main("C:/Users/eaffu/Documents/Personal_Projects/BankChurners.csv")


--- Overall ---


Attrition_Flag,Existing (%),Attrited (%)
proportion,83.934,16.066



--- Customer_Age ---


Attrition_Flag,Existing (%),Attrited (%)
Customer_Age_bin,,
"(25.999, 36.0]",87.1930,12.8070
"(36.0, 39.0]",85.2679,14.7321
"(39.0, 42.0]",82.6758,17.3242
"(42.0, 44.0]",82.6310,17.3690
"(44.0, 46.0]",83.5041,16.4959
"(46.0, 48.0]",83.0705,16.9295
"(48.0, 51.0]",84.5353,15.4647
"(51.0, 53.0]",84.6658,15.3342
"(53.0, 57.0]",81.6993,18.3007



--- Dependent_count ---


...

Attrition_Flag,Existing (%),Attrited (%)
Total_Trans_Ct_bin,,
"(9.999, 33.0]",73.3792,26.6208
"(33.0, 41.0]",61.5312,38.4688
"(41.0, 50.0]",51.6160,48.3840
"(50.0, 61.0]",79.0743,20.9257
"(61.0, 67.0]",91.8580,8.1420
"(67.0, 73.0]",94.6381,5.3619
"(73.0, 78.0]",95.5010,4.4990
"(78.0, 83.0]",97.8632,2.1368
"(83.0, 92.0]",97.9654,2.0346



--- Total_Ct_Chng_Q4_Q1 ---


Attrition_Flag,Existing (%),Attrited (%)
Total_Ct_Chng_Q4_Q1_bin,,
"(-0.001, 0.452]",43.9842,56.0158
"(0.452, 0.545]",71.7092,28.2908
"(0.545, 0.609]",81.3477,18.6523
"(0.609, 0.657]",88.1474,11.8526
"(0.657, 0.702]",92.7308,7.2692
"(0.702, 0.745]",91.9052,8.0948
"(0.745, 0.791]",91.8408,8.1592
"(0.791, 0.853]",92.0949,7.9051
"(0.853, 0.95]",92.9703,7.0297



--- Avg_Utilization_Ratio ---


Attrition_Flag,Existing (%),Attrited (%)
Avg_Utilization_Ratio_bin,,
"(-0.001, 0.051]",67.5658,32.4342
"(0.051, 0.1]",91.8065,8.1935
"(0.1, 0.176]",90.5605,9.4395
"(0.176, 0.277]",90.5660,9.4340
"(0.277, 0.43]",89.4995,10.5005
"(0.43, 0.573]",93.5388,6.4612
"(0.573, 0.707]",93.9783,6.0217
"(0.707, 0.999]",86.7589,13.2411


In [ ]:
#############
# SPLIT DATA
#############
# split the data into features (X) and target (y)
# axes=1 makes sure that a column is dropped and not a row
# test_size=0.2 means that 20% of the data is used for testing, 80% is used for training
# random_state=42 ensures reproducibility. Every time I run this code, I will get the same split
# stratify=y ensures that the proportion of attrited vs existing customers is the same in both training and test sets
# X_train = features for training
# X_test = features for testing
# y_train = target for training
# y_test = target for testing
def split_data(df):
    X = df.drop('Attrition_Flag', axis=1)
    y = df['Attrition_Flag']
    return train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

#####################
# TRAIN THE ML MODELS
#####################
# name = the name of the model
# model = the machine learning model object
# param_grid = dictionary of hyperparameters to search over using GridSearchCV
# scale = whether to apply StandardScaler to scale features
def train_model(name, model, param_grid, X_train, y_train, X_test, y_test, scale=True):
    print(f"\nTraining: {name}")
    
    # steps, hold the sequence of transformations and the model for a Pipeline
    # pipeline, allows one to combine preprocessing and model training in one object, everything happens in the correct order.
    # StandardScaler standardizes numeric features to have mean = 0 and st dev = 1
    # add ML model to the pipeline
    steps = []
    if scale:
        steps.append(('scaler', StandardScaler()))
    steps.append(('model', model))
    
    # Pipeline chains together multiple steps into one object
    # GridSearchCV automates hyperparameter tuning
    # cv is cross validation, I set it to 5
    # grid.fit trains the pipeline for all hyperparameter combinations using the training data 
    pipeline = Pipeline(steps)
    grid = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')
    grid.fit(X_train, y_train)
    
    # grid.best_estimator_ gives the pipeline that performed best during cross validation
    # predict() generates predictions for the features in the text sample
    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    
    # hasattr checks whether the model has a predict_proba method
    
    # predict_proba(X_test) returns probabilities of 0 and 1 
    # [:,1] selects only the probability of attrition   
    # if the model doesn’t have predict_proba, it cannot calculate probabilities, use y_pred as proxy
    # In that case, we just use the predicted class labels (y_pred) as a proxy.
    if hasattr(best_model.named_steps['model'], "predict_proba"):
        y_prob = best_model.predict_proba(X_test)[:,1]
    else:
        y_prob = y_pred
    
    print("Best Params:", grid.best_params_)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_prob))
    print(classification_report(y_test, y_pred))
    
    return best_model

# define all ML models to be trained - model object, hyperparameters, whether feature scaling is required
def get_models():
       return {
        "Logistic Regression": {
            "model": LogisticRegression(max_iter=1000, class_weight='balanced'),
            "params": {"model__C": [0.01, 0.1, 1, 10]},
            "scale": True  # scaling needed for linear models
        },

        "Decision Tree": {
            "model": DecisionTreeClassifier(),
            "params": {
                "model__max_depth": [3, 5, 10, None],
                "model__min_samples_split": [2, 5, 10]
            },
            "scale": False  # trees do not need scaling
        },

        "Random Forest": {
            "model": RandomForestClassifier(class_weight='balanced'),
            "params": {
                "model__n_estimators": [100, 200],
                "model__max_depth": [5, 10, None]
            },
            "scale": False
        },

        "Gradient Boosting": {
            "model": GradientBoostingClassifier(),
            "params": {
                "model__n_estimators": [100, 200],
                "model__learning_rate": [0.01, 0.1],
                "model__max_depth": [3, 5]
            },
            "scale": False
        },

        "AdaBoost": {
            "model": AdaBoostClassifier(),
            "params": {
                "model__n_estimators": [50, 100],
                "model__learning_rate": [0.01, 0.1, 1]
            },
            "scale": False
        },

        "SVM": {
            "model": SVC(probability=True),
            "params": {
                "model__C": [0.1, 1, 10],
                "model__kernel": ["rbf", "linear"]
            },
            "scale": True
        },

        "KNN": {
            "model": KNeighborsClassifier(),
            "params": {
                "model__n_neighbors": [3, 5, 7],
                "model__weights": ["uniform", "distance"]
            },
            "scale": True
        },

        "Naive Bayes": {
            "model": GaussianNB(),
            "params": {"model__var_smoothing": [1e-9, 1e-8, 1e-7]},
            "scale": False
        }
    }

def run_all_models(models, X_train, y_train, X_test, y_test):
    results = []
    trained_models = {}
    for name, config in models.items():
        model = train_model(name, config["model"], config["params"], X_train, y_train, X_test, y_test, scale=config["scale"])
        trained_models[name] = model
        y_pred = model.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        results.append((name, acc))
    results_df = pd.DataFrame(results, columns=['Model','Accuracy']).sort_values(by='Accuracy', ascending=False)
    return trained_models, results_df

################
# MAIN - MODELS
################
# Input data path
path = "C:/Users/eaffu/Documents/Personal_Projects/BankChurners.csv"

# Load data
df = load_data(path)

# Split data
X_train, X_test, y_train, y_test = split_data(df)

# Train models and compare
models = get_models()
trained_models, results_df = run_all_models(models, X_train, y_train, X_test, y_test)

print("\nModel Comparison:")
display(results_df)


Training: Logistic Regression
Best Params: {'model__C': 0.1}
Accuracy: 0.8558736426456071
ROC-AUC: 0.9196581196581196
              precision    recall  f1-score   support

           0       0.96      0.86      0.91      1701
           1       0.53      0.82      0.65       325

    accuracy                           0.86      2026
   macro avg       0.75      0.84      0.78      2026
weighted avg       0.89      0.86      0.87      2026


Training: Decision Tree
Best Params: {'model__max_depth': 10, 'model__min_samples_split': 5}
Accuracy: 0.9437314906219151
ROC-AUC: 0.9205209605209603
              precision    recall  f1-score   support

           0       0.96      0.97      0.97      1701
           1       0.84      0.80      0.82       325

    accuracy                           0.94      2026
   macro avg       0.90      0.88      0.89      2026
weighted avg       0.94      0.94      0.94      2026


Training: Random Forest
Best Params: {'model__max_depth': 10, 'model__n_est